In [0]:
from datetime import date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
hist = [
    ("Alice", 34, "HR"),
    ("Bob", 45, "Finance"),
    ("Cathy", 29, "IT")
]
schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True)
])

df_hist = spark.createDataFrame(hist,schema)

In [0]:
from datetime import date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
inc = [
    ("Alice", 100, "HR"),
    ("Boby", 45, "Finance"),
    ("swe", 29, "IT")
]
schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True)
])
df_inc = spark.createDataFrame(inc,schema)


In [0]:
df_inc_hashed=df_inc.withColumn('hashed',F.md5(F.concat_ws("||",F.coalesce(F.col('Name'),F.lit("")),F.coalesce(F.col('Age'),F.lit("")),F.coalesce(F.col('Department'),F.lit("")))))

In [0]:
df_inc.write.format('Delta').mode('append').saveAsTable('SCD2.BR.tb')

In [0]:
from pyspark.sql import functions as F
df_hist_hashed=df_hist.withColumn('hashed',F.md5(F.concat_ws("||",F.coalesce(F.col('Name'),F.lit("")),F.coalesce(F.col('Age'),F.lit("")),F.coalesce(F.col('Department'),F.lit("")))))

In [0]:
df_inc_hashed.write.format('Delta').mode('append').saveAsTable('SCD2.BR.tb_hashed')

In [0]:
%sql
select * from SCD2.BR.tb_hashed

In [0]:
df_hist_ha = spark.sql("SELECT * FROM SCD2.BR.tb_hashed")

In [0]:
df_hist_ha.createOrReplaceTempView("view_brhashed")

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS default.silver_customer (
        customer_id INT,
        email STRING,
        status STRING,
        row_hash STRING,
        start_date TIMESTAMP,
        end_date TIMESTAMP,
        is_current BOOLEAN
    ) USING DELTA
""")

In [0]:
from pyspark.sql.types import TimestampType
df_silver_init = df_hist \
    .withColumn("row_hash", F.md5(F.concat_ws("||", F.coalesce(F.col("Age"), F.lit("")), F.coalesce(F.col("Department"), F.lit(""))))) \
    .withColumn("start_date", F.to_timestamp(F.lit("2026-01-01 00:00:00"))) \
    .withColumn("end_date", F.lit(None).cast(TimestampType())) \
    .withColumn("is_current", F.lit(True))

In [0]:
from pyspark.sql import functions as F
df_silver=spark.sql("""Select * from scd2.sil.sil""")
df_silver=df_silver.withColumn('id',F.monotonically_increasing_id())

In [0]:
df_silver_init.show()

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("default.silver_employee")

In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema",True).saveAsTable("scd2.sil.sil")

In [0]:
%sql
select * from view_brhashed

In [0]:
scd2_stage= spark.sql(
    """select b.name as merge_key,
    b.name,b.age,b.department,b.hashed 
    from view_brhashed b

    union all

    select null as merge_key,
    a.name,a.age,a.department,a.hashed 
    from view_brhashed a
    join scd2.sil.sil s
    on a.name = s.name
    and s.is_current = true
    where a.hashed != s.row_hash
    """
)

In [0]:
scd2_stage.createOrReplaceTempView("scd2_v")

In [0]:
scd2_stage.show()

In [0]:
spark.sql(
    """
    Merge into scd2.sil.sil t
    using scd2_v s
    on t.name = s.merge_key
    and t.is_current = true

    when matched and t.row_hash <> s.hashed then
    update set t.end_date = current_timestamp(),
                t.is_current = false

    when not matched then
    insert (name,age,department,row_hash,start_date,end_date,is_current,id)
    values (s.name,s.age,s.department,s.hashed,current_timestamp(),null,true,default)
    """
)

# Enable Liquid Clustering on name and is_current columns
spark.sql("""
    ALTER TABLE scd2.sil.sil 
    CLUSTER BY (name, is_current)
""")

In [0]:
%sql
Select * from scd2.sil.sil

In [0]:
%sql
Delete from scd2.sil.sil where id in(
    select id from(
        select id,row_number() OVER(PARTITION BY name order by end_date desc) as rn from scd2.sil.sil
        where is_current = false
    )tmp
    where rn>1
)

In [0]:
%sql
DESCRIBE HISTORY scd2.sil.sil;


In [0]:
df_old = spark.read.format("delta").option("versionAsOf", 3).table("scd2.sil.sil")

In [0]:
df_old.show()